# Personal Finance Tracker Analysis

This notebook analyzes category-level spending patterns and segments users by savings behavior.
The same data model is used by `finance_story_dashboard.html`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)
DATA_FILE = Path("personal_finance_tracker_dataset.csv")
OUTPUT_DIR = Path("analysis_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load and Inspect the Dataset

In [2]:
df = pd.read_csv(DATA_FILE, parse_dates=["date"])
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()
print(f"Rows: {len(df):,}")
print(f"Users: {df['user_id'].nunique():,}")
print(f"Period: {df['date'].min():%Y-%m-%d} to {df['date'].max():%Y-%m-%d}")
display(df.head())
display(df.isna().sum().to_frame("missing_values"))

Rows: 3,000
Users: 944
Period: 2019-01-01 to 2023-11-06


,date,user_id,monthly_income,monthly_expense_total,savings_rate,budget_goal,financial_scenario,credit_score,debt_to_income_ratio,loan_payment,investment_amount,subscription_services,emergency_fund,transaction_count,fraud_flag,discretionary_spending,essential_spending,income_type,rent_or_mortgage,category,cash_flow_status,financial_advice_score,financial_stress_level,actual_savings,savings_goal_met,month
0,2019-01-01,1584,3119.58,3212.07,0.38,3676.11,inflation,721.0,0.56,125.77,689.22,3,510.58,68,0,857.55,1910.85,Freelance,1501.65,Investments,Positive,8.3,Low,0.00,0,2019-01-01
1,2019-01-31,1045,3262.44,3732.81,0.10,2607.17,inflation,670.0,0.42,454.19,360.34,4,1154.41,41,0,534.51,3165.20,Salary,1603.17,Investments,Positive,22.6,Low,0.00,0,2019-01-01
2,2019-03-02,1756,2931.20,3335.58,0.15,3004.14,inflation,691.0,0.24,971.82,0.00,5,1433.02,90,0,353.67,1504.56,Freelance,1097.82,Healthcare,Positive,58.8,Low,0.00,0,2019-03-01
3,2019-04-01,1724,3506.79,2327.59,0.17,3346.97,normal,717.0,0.16,482.76,182.06,5,227.37,94,0,594.08,1450.72,Freelance,1155.64,Groceries,Positive,74.5,Low,1179.20,0,2019-04-01
4,2019-05-01,1600,4606.87,2182.58,0.34,2670.09,inflation,795.0,0.25,263.74,342.78,9,589.81,73,0,556.86,1000.00,Salary,1170.86,Utilities,Negative,38.7,High,2424.29,0,2019-05-01


,missing_values
date,0
user_id,0
monthly_income,0
monthly_expense_total,0
savings_rate,0
budget_goal,0
financial_scenario,0
credit_score,0
debt_to_income_ratio,0
loan_payment,0


## 2. Feature Engineering

In [3]:
stress_map = {"Low": 1, "Medium": 2, "High": 3}

df["expense_to_income"] = df["monthly_expense_total"] / df["monthly_income"]
df["discretionary_share"] = df["discretionary_spending"] / df["monthly_expense_total"]
df["essential_share"] = df["essential_spending"] / df["monthly_expense_total"]
df["investment_rate"] = df["investment_amount"] / df["monthly_income"]
df["emergency_fund_months"] = df["emergency_fund"] / df["monthly_expense_total"]
df["stress_score"] = df["financial_stress_level"].map(stress_map)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
ratio_cols = [
    "expense_to_income", "discretionary_share", "essential_share",
    "investment_rate", "emergency_fund_months"
]
df[ratio_cols] = df[ratio_cols].fillna(0)
display(df[ratio_cols + ["stress_score"]].describe().T)

,count,mean,std,min,25%,50%,75%,max
expense_to_income,3000.0,0.814007,0.369109,0.034643,0.583216,0.749035,0.956801,5.468889
discretionary_share,3000.0,0.183006,0.111057,0.000000,0.115231,0.166116,0.227294,2.553357
essential_share,3000.0,0.811418,0.508227,0.198323,0.558886,0.733795,0.949724,18.965140
investment_rate,3000.0,0.109076,0.080683,0.000000,0.057233,0.097832,0.145516,0.962982
emergency_fund_months,3000.0,0.367196,0.249904,0.000000,0.217526,0.334715,0.471112,5.347152
stress_score,3000.0,1.690667,0.780502,1.000000,1.000000,1.000000,2.000000,3.000000


## 3. Pattern Analysis by Category

In [4]:
category_summary = (
    df.groupby("category", as_index=False)
    .agg(
        records=("category", "size"),
        avg_income=("monthly_income", "mean"),
        avg_expense=("monthly_expense_total", "mean"),
        avg_actual_savings=("actual_savings", "mean"),
        avg_savings_rate=("savings_rate", "mean"),
        goal_met_rate=("savings_goal_met", "mean"),
        avg_debt_to_income=("debt_to_income_ratio", "mean"),
        avg_discretionary_share=("discretionary_share", "mean"),
        avg_essential_share=("essential_share", "mean"),
        avg_stress_score=("stress_score", "mean"),
        fraud_rate=("fraud_flag", "mean"),
    )
    .sort_values("avg_expense", ascending=False)
)
category_summary["record_share"] = category_summary["records"] / category_summary["records"].sum()
category_summary.to_csv(OUTPUT_DIR / "category_summary.csv", index=False)

display(category_summary.style.format({
    "avg_income": "${:,.0f}",
    "avg_expense": "${:,.0f}",
    "avg_actual_savings": "${:,.0f}",
    "avg_savings_rate": "{:.1%}",
    "goal_met_rate": "{:.1%}",
    "avg_debt_to_income": "{:.1%}",
    "avg_discretionary_share": "{:.1%}",
    "avg_essential_share": "{:.1%}",
    "fraud_rate": "{:.1%}",
    "record_share": "{:.1%}",
}))

,category,records,avg_income,avg_expense,avg_actual_savings,avg_savings_rate,goal_met_rate,avg_debt_to_income,avg_discretionary_share,avg_essential_share,avg_stress_score,fraud_rate,record_share
3,Groceries,308,"$4,000","$3,076","$1,060",23.1%,6.2%,34.0%,17.6%,78.6%,1.623377,3.2%,10.3%
5,Insurance,321,"$4,020","$3,070","$1,133",22.8%,9.7%,35.6%,17.9%,80.0%,1.753894,2.8%,10.7%
0,Dining Out,306,"$3,950","$3,065","$1,076",22.4%,10.8%,35.3%,17.7%,80.0%,1.699346,1.3%,10.2%
9,Utilities,320,"$3,902","$3,050","$1,057",22.7%,6.9%,34.8%,18.0%,80.2%,1.659375,1.6%,10.7%
2,Entertainment,291,"$3,994","$3,041","$1,145",23.5%,7.9%,35.5%,19.2%,84.5%,1.773196,2.7%,9.7%
4,Healthcare,308,"$4,009","$2,995","$1,149",22.4%,6.8%,34.6%,18.4%,82.8%,1.616883,2.6%,10.3%
8,Transportation,305,"$4,066","$2,969","$1,265",22.1%,11.5%,34.7%,18.8%,81.7%,1.711475,2.6%,10.2%
6,Investments,286,"$4,102","$2,961","$1,275",22.4%,11.5%,34.5%,18.8%,81.6%,1.678322,1.7%,9.5%
1,Education,289,"$4,004","$2,940","$1,206",22.6%,10.4%,35.9%,18.4%,80.8%,1.698962,2.1%,9.6%
7,Rent,266,"$4,004","$2,929","$1,223",21.7%,11.3%,35.8%,18.4%,81.5%,1.695489,3.0%,8.9%


In [5]:
fig = px.bar(
    category_summary.sort_values("avg_expense"),
    x="avg_expense",
    y="category",
    color="avg_savings_rate",
    color_continuous_scale=["#f5b7a7", "#f6d365", "#82c0cc"],
    labels={
        "avg_expense": "Average monthly expense",
        "category": "",
        "avg_savings_rate": "Savings rate",
    },
    title="Average Monthly Expense by Category"
)
fig.update_xaxes(tickprefix="$", separatethousands=True)
fig.show()

In [6]:
fig = px.bar(
    category_summary.sort_values("goal_met_rate", ascending=False),
    x="category",
    y="goal_met_rate",
    color="avg_debt_to_income",
    color_continuous_scale=["#7fc97f", "#fdc086", "#beaed4"],
    labels={
        "category": "",
        "goal_met_rate": "Savings goal met rate",
        "avg_debt_to_income": "Debt-to-income",
    },
    title="Savings Goal Achievement by Category"
)
fig.update_yaxes(tickformat=".0%")
fig.show()

## 4. Savings Behavior Segmentation

In [7]:
user = (
    df.groupby("user_id", as_index=False)
    .agg(
        observations=("user_id", "size"),
        avg_income=("monthly_income", "mean"),
        avg_expense=("monthly_expense_total", "mean"),
        avg_savings_rate=("savings_rate", "mean"),
        avg_actual_savings=("actual_savings", "mean"),
        goal_met_rate=("savings_goal_met", "mean"),
        avg_debt_to_income=("debt_to_income_ratio", "mean"),
        avg_loan_payment=("loan_payment", "mean"),
        avg_investment_rate=("investment_rate", "mean"),
        avg_emergency_fund_months=("emergency_fund_months", "mean"),
        avg_discretionary_share=("discretionary_share", "mean"),
        avg_expense_to_income=("expense_to_income", "mean"),
        high_stress_rate=("financial_stress_level", lambda s: (s == "High").mean()),
    )
)

segment_features = [
    "avg_income", "avg_expense", "avg_savings_rate", "avg_actual_savings",
    "goal_met_rate", "avg_debt_to_income", "avg_loan_payment",
    "avg_investment_rate", "avg_emergency_fund_months",
    "avg_discretionary_share", "high_stress_rate", "avg_expense_to_income"
]
scaled = StandardScaler().fit_transform(user[segment_features])
user["cluster"] = KMeans(n_clusters=4, n_init=25, random_state=42).fit_predict(scaled)

In [8]:
cluster_summary = (
    user.groupby("cluster", as_index=False)
    .agg(
        users=("user_id", "size"),
        avg_income=("avg_income", "mean"),
        avg_expense=("avg_expense", "mean"),
        avg_savings_rate=("avg_savings_rate", "mean"),
        avg_actual_savings=("avg_actual_savings", "mean"),
        goal_met_rate=("goal_met_rate", "mean"),
        avg_debt_to_income=("avg_debt_to_income", "mean"),
        avg_investment_rate=("avg_investment_rate", "mean"),
        avg_emergency_fund_months=("avg_emergency_fund_months", "mean"),
        avg_discretionary_share=("avg_discretionary_share", "mean"),
        high_stress_rate=("high_stress_rate", "mean"),
        avg_expense_to_income=("avg_expense_to_income", "mean"),
    )
)

metrics = [
    "avg_savings_rate", "goal_met_rate", "avg_emergency_fund_months",
    "avg_investment_rate", "avg_debt_to_income", "avg_expense_to_income"
]
z = cluster_summary[metrics].apply(lambda s: (s - s.mean()) / (s.std(ddof=0) or 1))
cluster_summary["strength_score"] = (
    z["avg_savings_rate"] + z["goal_met_rate"] + z["avg_emergency_fund_months"]
    + z["avg_investment_rate"] - z["avg_debt_to_income"] - z["avg_expense_to_income"]
)

labels = {}
available = set(cluster_summary["cluster"])
strongest = int(cluster_summary.loc[cluster_summary["strength_score"].idxmax(), "cluster"])
labels[strongest] = "Resilient investors"
available.remove(strongest)
goal_cluster = int(cluster_summary[cluster_summary["cluster"].isin(available)].sort_values(
    ["goal_met_rate", "avg_savings_rate"], ascending=False
).iloc[0]["cluster"])
labels[goal_cluster] = "Goal-focused savers"
available.remove(goal_cluster)
debt_cluster = int(cluster_summary[cluster_summary["cluster"].isin(available)].sort_values(
    ["avg_debt_to_income", "avg_expense_to_income"], ascending=False
).iloc[0]["cluster"])
labels[debt_cluster] = "Debt-pressure spenders"
available.remove(debt_cluster)
for cluster in available:
    labels[int(cluster)] = "Cash-flow stretched"

user["savings_segment"] = user["cluster"].map(labels)
segment_summary = (
    user.groupby("savings_segment", as_index=False)
    .agg(
        users=("user_id", "size"),
        avg_income=("avg_income", "mean"),
        avg_expense=("avg_expense", "mean"),
        avg_savings_rate=("avg_savings_rate", "mean"),
        avg_actual_savings=("avg_actual_savings", "mean"),
        goal_met_rate=("goal_met_rate", "mean"),
        avg_debt_to_income=("avg_debt_to_income", "mean"),
        avg_investment_rate=("avg_investment_rate", "mean"),
        avg_emergency_fund_months=("avg_emergency_fund_months", "mean"),
        avg_discretionary_share=("avg_discretionary_share", "mean"),
        high_stress_rate=("high_stress_rate", "mean"),
        avg_expense_to_income=("avg_expense_to_income", "mean"),
    )
    .sort_values("users", ascending=False)
)
segment_summary["user_share"] = segment_summary["users"] / segment_summary["users"].sum()

user.to_csv(OUTPUT_DIR / "user_savings_segments.csv", index=False)
segment_summary.to_csv(OUTPUT_DIR / "segment_summary.csv", index=False)
display(segment_summary.style.format({
    "avg_income": "${:,.0f}",
    "avg_expense": "${:,.0f}",
    "avg_actual_savings": "${:,.0f}",
    "avg_savings_rate": "{:.1%}",
    "goal_met_rate": "{:.1%}",
    "avg_debt_to_income": "{:.1%}",
    "avg_investment_rate": "{:.1%}",
    "avg_emergency_fund_months": "{:.2f}",
    "avg_discretionary_share": "{:.1%}",
    "high_stress_rate": "{:.1%}",
    "avg_expense_to_income": "{:.1%}",
    "user_share": "{:.1%}",
}))

,savings_segment,users,avg_income,avg_expense,avg_savings_rate,avg_actual_savings,goal_met_rate,avg_debt_to_income,avg_investment_rate,avg_emergency_fund_months,avg_discretionary_share,high_stress_rate,avg_expense_to_income,user_share
1,Debt-pressure spenders,411,"$4,149","$3,256",21.6%,"$1,009",4.4%,36.8%,9.7%,0.31,15.5%,17.4%,82.0%,43.5%
3,Resilient investors,252,"$3,982","$2,553",24.5%,"$1,487",7.2%,33.1%,11.2%,0.45,23.1%,18.2%,67.3%,26.7%
0,Cash-flow stretched,174,"$3,200","$3,330",22.8%,$370,1.2%,32.9%,14.9%,0.31,15.7%,20.8%,114.1%,18.4%
2,Goal-focused savers,107,"$4,965","$2,542",21.1%,"$2,435",51.7%,36.1%,7.7%,0.49,21.5%,23.7%,53.1%,11.3%


### Segment Meaning, Standards, and Recommended Actions

Each segment name is an interpretation of a KMeans cluster. The standard combines a practical benchmark
for how the segment should be managed with the actual data profile observed in this dataset.

In [9]:
segment_profiles = {
    "Resilient investors": {
        "meaning": "Strongest overall savings profile, with healthier savings, emergency coverage, and manageable debt pressure.",
        "standard": "Benchmark segment: keep savings consistent, preserve liquidity, and deepen investment planning without adding avoidable debt.",
        "bank_action": "Offer high-yield savings, automated investing, portfolio review, and pre-approved credit only when repayment capacity remains healthy.",
        "advisor_action": "Focus on wealth building: asset allocation, tax-aware investing, insurance coverage, and retirement milestones.",
    },
    "Goal-focused savers": {
        "meaning": "Most likely to hit savings goals, showing target-based discipline even when average savings rate is not the highest.",
        "standard": "Goal-achievement standard: protect the habit with automatic transfers, target dates, and alerts when spending threatens the goal.",
        "bank_action": "Provide goal pockets, round-up savings, bonus rates for streaks, and personalized goal-progress nudges.",
        "advisor_action": "Translate goals into a plan: emergency fund target, debt payoff schedule, investment contribution ladder, and review cadence.",
    },
    "Debt-pressure spenders": {
        "meaning": "Highest debt burden and lower goal achievement. Cash flow exists, but debt payments and expenses limit progress.",
        "standard": "Risk-reduction standard: bring debt-to-income down first, then rebuild savings goals after repayment pressure eases.",
        "bank_action": "Prioritize refinance checks, debt consolidation options, payment reminders, spending limits, and restructuring support where suitable.",
        "advisor_action": "Build a debt-first plan: rank balances by rate, cap discretionary leakage, protect minimum emergency cash, and set payoff milestones.",
    },
    "Cash-flow stretched": {
        "meaning": "Expenses are close to or above income, leaving little room for actual savings despite moderate savings-rate signals.",
        "standard": "Stabilization standard: create positive monthly cash flow before pushing aggressive investment or long-term saving targets.",
        "bank_action": "Use low-balance alerts, bill timing tools, subscription review, overdraft prevention, and small emergency savings automation.",
        "advisor_action": "Start with cash-flow repair: separate essential and discretionary spending, reset the budget, renegotiate recurring costs, and define a starter emergency fund.",
    },
}

guidance_rows = []
summary_lookup = {
    row["savings_segment"]: row for row in segment_summary.to_dict("records")
}
for segment_name, profile in segment_profiles.items():
    if segment_name not in summary_lookup:
        continue
    row = summary_lookup[segment_name]
    guidance_rows.append({
        "savings_segment": segment_name,
        "users": int(row["users"]),
        "user_share": row["user_share"],
        "meaning": profile["meaning"],
        "standard": profile["standard"],
        "data_standard": (
            f"{row['avg_savings_rate']:.1%} avg savings rate; "
            f"{row['goal_met_rate']:.1%} goal-met rate; "
            f"{row['avg_debt_to_income']:.1%} debt-to-income; "
            f"{row['avg_emergency_fund_months']:.2f} months emergency coverage."
        ),
        "bank_action": profile["bank_action"],
        "advisor_action": profile["advisor_action"],
    })

segment_guidance = pd.DataFrame(guidance_rows)
segment_guidance.to_csv(OUTPUT_DIR / "segment_guidance.csv", index=False)
display(segment_guidance[[
    "savings_segment", "users", "user_share", "meaning", "standard", "data_standard"
]].style.format({"user_share": "{:.1%}"}))

,savings_segment,users,user_share,meaning,standard,data_standard
0,Resilient investors,252,26.7%,"Strongest overall savings profile, with healthier savings, emergency coverage, and manageable debt pressure.","Benchmark segment: keep savings consistent, preserve liquidity, and deepen investment planning without adding avoidable debt.",24.5% avg savings rate; 7.2% goal-met rate; 33.1% debt-to-income; 0.45 months emergency coverage.
1,Goal-focused savers,107,11.3%,"Most likely to hit savings goals, showing target-based discipline even when average savings rate is not the highest.","Goal-achievement standard: protect the habit with automatic transfers, target dates, and alerts when spending threatens the goal.",21.1% avg savings rate; 51.7% goal-met rate; 36.1% debt-to-income; 0.49 months emergency coverage.
2,Debt-pressure spenders,411,43.5%,"Highest debt burden and lower goal achievement. Cash flow exists, but debt payments and expenses limit progress.","Risk-reduction standard: bring debt-to-income down first, then rebuild savings goals after repayment pressure eases.",21.6% avg savings rate; 4.4% goal-met rate; 36.8% debt-to-income; 0.31 months emergency coverage.
3,Cash-flow stretched,174,18.4%,"Expenses are close to or above income, leaving little room for actual savings despite moderate savings-rate signals.",Stabilization standard: create positive monthly cash flow before pushing aggressive investment or long-term saving targets.,22.8% avg savings rate; 1.2% goal-met rate; 32.9% debt-to-income; 0.31 months emergency coverage.


### What Banks and Personal Finance Advisors Can Do

In [10]:
display(segment_guidance[[
    "savings_segment", "bank_action", "advisor_action"
]])

,savings_segment,bank_action,advisor_action
0,Resilient investors,"Offer high-yield savings, automated investing,...","Focus on wealth building: asset allocation, ta..."
1,Goal-focused savers,"Provide goal pockets, round-up savings, bonus ...",Translate goals into a plan: emergency fund ta...
2,Debt-pressure spenders,"Prioritize refinance checks, debt consolidatio...",Build a debt-first plan: rank balances by rate...
3,Cash-flow stretched,"Use low-balance alerts, bill timing tools, sub...",Start with cash-flow repair: separate essentia...


In [11]:
colors = {
    "Resilient investors": "#197278",
    "Goal-focused savers": "#2f80ed",
    "Debt-pressure spenders": "#c44536",
    "Cash-flow stretched": "#7b61ff",
}
fig = px.scatter(
    user,
    x="avg_debt_to_income",
    y="avg_savings_rate",
    size="avg_income",
    color="savings_segment",
    color_discrete_map=colors,
    hover_data={
        "user_id": True,
        "avg_income": ":$,.0f",
        "avg_expense": ":$,.0f",
        "goal_met_rate": ":.0%",
        "avg_emergency_fund_months": ":.2f",
    },
    labels={
        "avg_debt_to_income": "Average debt-to-income",
        "avg_savings_rate": "Average savings rate",
        "savings_segment": "Segment",
    },
    title="User Savings Behavior Segments"
)
fig.update_xaxes(tickformat=".0%")
fig.update_yaxes(tickformat=".0%")
fig.show()

## 5. Trend and Scenario Context

In [12]:
monthly_summary = (
    df.groupby("month", as_index=False)
    .agg(
        avg_income=("monthly_income", "mean"),
        avg_expense=("monthly_expense_total", "mean"),
        avg_actual_savings=("actual_savings", "mean"),
        avg_savings_rate=("savings_rate", "mean"),
        goal_met_rate=("savings_goal_met", "mean"),
        avg_debt_to_income=("debt_to_income_ratio", "mean"),
    )
    .sort_values("month")
)
monthly_summary.to_csv(OUTPUT_DIR / "monthly_summary.csv", index=False)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=monthly_summary["month"], y=monthly_summary["avg_savings_rate"],
    mode="lines", name="Savings rate", line=dict(color="#197278", width=3)
))
fig.add_trace(go.Scatter(
    x=monthly_summary["month"], y=monthly_summary["goal_met_rate"],
    mode="lines", name="Goal met rate", line=dict(color="#f2a541", width=3)
))
fig.update_layout(title="Savings Discipline Over Time", legend=dict(orientation="h"))
fig.update_yaxes(tickformat=".0%")
fig.show()

In [13]:
scenario_category = (
    df.groupby(["financial_scenario", "category"], as_index=False)
    .agg(avg_savings_rate=("savings_rate", "mean"))
)
heat = scenario_category.pivot(
    index="financial_scenario", columns="category", values="avg_savings_rate"
)
fig = px.imshow(
    heat,
    color_continuous_scale=["#c44536", "#f2a541", "#197278"],
    aspect="auto",
    labels=dict(color="Savings rate", x="", y=""),
    title="Savings Rate by Category and Financial Scenario"
)
fig.update_coloraxes(colorbar_tickformat=".0%")
fig.show()

## 6. Executive Takeaways

In [14]:
highest_expense = category_summary.iloc[0]
best_goal = category_summary.sort_values("goal_met_rate", ascending=False).iloc[0]
weakest_goal = category_summary.sort_values("goal_met_rate").iloc[0]
largest_segment = segment_summary.sort_values("users", ascending=False).iloc[0]
strongest_segment = segment_summary.sort_values("avg_savings_rate", ascending=False).iloc[0]
debt_segment = segment_summary.sort_values("avg_debt_to_income", ascending=False).iloc[0]

takeaways = [
    f"{highest_expense['category']} has the highest average monthly expense at ${highest_expense['avg_expense']:,.0f}.",
    f"{best_goal['category']} has the strongest savings goal performance at {best_goal['goal_met_rate']:.1%}; {weakest_goal['category']} is lowest at {weakest_goal['goal_met_rate']:.1%}.",
    f"The largest savings segment is {largest_segment['savings_segment']}, covering {largest_segment['user_share']:.1%} of users.",
    f"{strongest_segment['savings_segment']} is the benchmark segment with a {strongest_segment['avg_savings_rate']:.1%} average savings rate.",
    f"{debt_segment['savings_segment']} carries the highest average debt-to-income ratio at {debt_segment['avg_debt_to_income']:.1%}.",
]
for item in takeaways:
    print("-", item)

- Groceries has the highest average monthly expense at $3,076.
- Investments has the strongest savings goal performance at 11.5%; Groceries is lowest at 6.2%.
- The largest savings segment is Debt-pressure spenders, covering 43.5% of users.
- Resilient investors is the benchmark segment with a 24.5% average savings rate.
- Debt-pressure spenders carries the highest average debt-to-income ratio at 36.8%.
